# Time Series Caching Verification

In [1]:
import sys, os, json
from datetime import date
sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env
from apps.agentic.core.series_cache import SeriesCache

set_anthropic_env(filedir='../../.keys')

## SeriesCache

In [3]:
# Initialize
SeriesCache.initialize()

# Put a test entry
fake_data = {
    "observations": [
        {"date": "2020-01-01", "value": "100.0"},
        {"date": "2020-02-01", "value": "101.5"},
    ]
}

cache_id = await SeriesCache.put(
    source="fred",
    native_id="TEST_SERIES",
    data=fake_data,
    observation_start="2020-01-01",
    observation_end="2020-02-01",
    observation_count=2,
    frequency="Monthly",
    units="Index",
    title="Test Series",
)
print(f"cache_id: {cache_id}")

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada
DEBUG:    SeriesCache: stored fred:TEST_SERIES → 19d593ca-f355-4da6-a764-a3d8d7bf761a
cache_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a


In [5]:
# Fetch by cache_id
by_id = await SeriesCache.get_by_cache_id(cache_id)
assert by_id is not None
print(f"get_by_cache_id: {by_id['native_id']} — {by_id['observation_count']} obs")

# Fetch by native_idassert by_id is not None
by_native = await SeriesCache.get_by_native_id("fred", "TEST_SERIES")
assert by_native is not None
print(f"get_by_native_id: {by_native['cache_id']} — expires {by_native['expires_at'].date()}")

# Confirm same cache_id
assert str(by_native['cache_id']) == cache_id
print("OK — both lookups return the same entry")

get_by_cache_id: TEST_SERIES — 2 obs
get_by_native_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a — expires 2026-05-26
OK — both lookups return the same entry
